# Plot anomalies.

Begin by installing necessary packages.

In [1]:
import sys
!{sys.executable} -m pip install --quiet requests numpy wnutils xmlcoll pandas seaborn matplotlib scipy

Next, import the necessary packages:

In [2]:
import os, io, requests
import numpy as np
import wnutils as wn
import xmlcoll.coll as xc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy as sp

from ipywidgets import interactive_output, HBox
import ipywidgets as widgets

## Preliminaries

Set some preliminary data and define some functions needed for the notebook.  Begin by choosing the plot theme.

In [3]:
sns.set_style("ticks")

Define a routine to extract all possible species names from the data.

In [4]:
def get_species(df):
    species_list = []
    cols = df.columns
    for col in cols:
        s = col.split("_")
        if s[0] == 'mu':
            if s[1] not in species_list:
                species_list.append(s[1])
    return species_list

Next define a routine to create latex names from the relevant species.

In [5]:
def get_latex_names(df):
    wb = wn.Base()
    return wb.get_latex_names(get_species(df))

Define the routine to convert the relevant pandas dataframe values to floats.

In [6]:
def convert_to_float(df):
    my_series = []
    for species in get_species(df):
        my_series.append('mu_' + species + '_value')
        my_series.append('mu_' + species + '_variability of sample')
    
    df[my_series] = df[my_series].apply(pd.to_numeric)

Define a routine to retrieve the normalizing species.

In [7]:
def get_normalizing_species(coll, df):
    result = {}
    my_species = get_species(df)
    props = coll.get_properties()
    for species in my_species:
        result[species] = props[('mu', species, 'normalizing species')]
    return result

Define a routine to get the molecular cloud and NRLEE mass fractions at Solar time.

In [8]:
def get_mc_and_nrlee_mass_fractions(h5, t_sol):
    props = h5.get_zone_properties_in_groups_as_floats(('molecular cloud', '0', '0'), ['time', 'zone mass', 'metallicity'])

    t = np.array(props['time'])

    i = []
    for j in range(t.shape[0]):
        if t[j] > (t_sol * 3.15e16):
            i.append(j)
        
    name_dict = {}
    for key, value in h5.get_nuclide_data().items():
        name_dict[value['index']] = key
 
    group = h5.get_iterable_groups()[i[0]]
    zone_list = h5.get_zone_labels_for_group(group)

    x_g = h5.get_group_mass_fractions(group)

    i_mc = zone_list.index(('molecular cloud', '0', '0'))
    i_nrlee = zone_list.index(('nrlee dust', '0', '0'))

    x_mc = {}
    x_nrlee = {}
    for i in range(x_g.shape[1]):
        x_mc[name_dict[i]] = x_g[i_mc, i]
        x_nrlee[name_dict[i]] = x_g[i_nrlee, i]
        
    return [x_mc, x_nrlee]

Define a routine to compute the isotopic anomalies that result from adding and subtract NRLEE dust.

In [9]:
def compute_mus(df_ci, x_mc, x_nrlee, species, f0, df_low = -1, df_high = 0):

    r_sp0_nrlee = x_nrlee[species[0]] / x_mc[species[0]]
    r_sp1_nrlee = x_nrlee[species[1]] / x_mc[species[1]]

    r_sp2_nrlee = x_nrlee[species[2]] / x_mc[species[2]]
    r_sp3_nrlee = x_nrlee[species[3]] / x_mc[species[3]]
    
    df = np.linspace(df_low, df_high, 1000)

    df /= (1 - f0)
    
    xc = (1 + df * (r_sp0_nrlee - 1)) / (1 + df * (r_sp1_nrlee - 1)) - 1
    yc = (1 + df * (r_sp2_nrlee - 1)) / (1 + df * (r_sp3_nrlee - 1)) - 1
    
    x_val = "mu_" + species[0] + "_value"
    y_val = "mu_" + species[2] + "_value"

    return 1.e6 * xc * (1 + df_ci[x_val] / 1.e6) + df_ci[x_val], 1.e6 * yc * (1 + df_ci[y_val] / 1.e6) + df_ci[y_val]

Define a routine to create the plot title string.

In [10]:
def make_title_str(df, x_species, y_species, l_names, mus, nc_only):
    
    x_val = "mu_" + x_species + "_value"
    y_val = "mu_" + y_species + "_value"

    df_sub = df.loc[(~pd.isnull(df[x_val]) & ~pd.isnull(df[y_val]))]

    if nc_only:
        df_data = df_sub.loc[df['type'] == 'NC']
    else:
        df_data = df_sub

    result_data = sp.stats.linregress(df_data[x_val], df_data[y_val])
    result_model = sp.stats.linregress(mus[0], mus[1])

    return \
        'Fit: $\\mu$ {:s} = ({:.4f} $\\pm$ {:.4f}) $\\times\\ \\mu$ {:s}  + ({:.4f} $\\pm$ {:.4f}), R = {:.4f}\n\
         Model: $\\mu$ {:s} = {:.4f} $\\times\\ \\mu$ {:s} + {:.4f}'.format(
            l_names[y_species], result_data.slope, result_data.stderr, l_names[x_species],
            result_data.intercept, result_data.intercept_stderr, result_data.rvalue,
            l_names[y_species], result_model.slope, l_names[x_species], result_model.intercept)

Define a routine to compute CAI B's.

In [11]:
def compute_cai_b(x_mc, x_nrlee, sp, f_ci, f_cai, g_n, g_nbar):

    r_sp_nrlee = x_nrlee[sp] / x_mc[sp]
    
    c_i = f_cai * ((g_n/g_nbar) * ((1 - f_ci)/(1 - f_cai)) - (f_ci / f_cai))
        
    return g_nbar * ((1 - f_cai)/(1 - f_ci)) * (1 + c_i * r_sp_nrlee)

Define a routine to compute mu's for CAI-like material.

In [12]:
def compute_mu_cai(df_ci, x_mc, x_nrlee, species, f_ci, f_cai,
                   g_x_n, g_x_nbar, g_y_n, g_y_nbar):
    
    b0 = compute_cai_b(x_mc, x_nrlee, species[0], f_ci, f_cai, g_x_n, g_x_nbar)
    b1 = compute_cai_b(x_mc, x_nrlee, species[1], f_ci, f_cai, g_x_n, g_x_nbar)
    b2 = compute_cai_b(x_mc, x_nrlee, species[2], f_ci, f_cai, g_y_n, g_y_nbar)
    b3 = compute_cai_b(x_mc, x_nrlee, species[3], f_ci, f_cai, g_y_n, g_y_nbar)
            
    return 1.e6 * (b0/b1 - 1), 1.e6 * (b2/b3 - 1)
            
    x_val = "mu_" + species[0] + "_value"
    y_val = "mu_" + species[2] + "_value"

    return xc * (1 + df_ci[x_val] / 1.e6) + df_ci[x_val], yc * (1 + df_ci[y_val] / 1.e6) + df_ci[y_val]

## Retrieve data

First retrieve the meteoritic data.

In [13]:
my_collection = xc.Collection()
#my_collection.update_from_xml(io.BytesIO(requests.get('https://osf.io/wj5rd/download').content))
my_collection.update_from_xml('Brad.xml')

Now obtain the data frame, convert the relevant species data to numeric values, and get latex names for species.  Also retrieve a dictionary of the normalizing species for the key isotopes.

In [14]:
df = my_collection.get_dataframe()
convert_to_float(df)
lnames = get_latex_names(df)
norm_dict = get_normalizing_species(my_collection, df)

Extract the NC and CC data along with the CI and bulk Earth data.

In [15]:
df_nc = df.loc[df['type'] == 'NC']
df_cc = df.loc[df['type'] == 'CC']
df_ci = df.loc["CI"]
df_earth = df.loc["Bulk Earth"]

Now retrieve the GCE model.

In [16]:
#!rm -fr gce.h5
#!curl -J -L -o gce.h5.gz https://osf.io/me9cr/download
#!gunzip gce.h5.gz

Load the model into a *wnutils* H5 class instance.

In [17]:
!gunzip gce.h5.gz
h5 = wn.h5.H5('gce.h5')

Set the time (in Gyr) at which to compute the anomalies (that is, the time of Solar System formation).  Retrieve the mass fractions at that time.

In [18]:
t_sol = 8
x_mc, x_nrlee = get_mc_and_nrlee_mass_fractions(h5, t_sol)

# Model for the NC-CC Dichotomy

Create the plot.  Set the parameters.  *f0* is the fraction of the mass contributed by NRLEE dust to CI. *df = f - f0*, where *f* is the fraction of the mass contributed by the NRLEE dust to the sample.  To save a copy of the figure as a pdf, type the name of the figure in the *figure name* field.  The figure will be saved locally (on Colab, click on the folder icon on the left to access the figure).

In [19]:
def plot_nc_from_ci(x_species, y_species, f_4, df_f_low, df_f_high, show_title, nc_only, figure_name):

    v_species = [x_species, norm_dict[x_species], y_species, norm_dict[y_species]]
    
    f = 1.e-4 * f_4
    df_low = df_f_low * f * 1.e-3
    df_high = df_f_high * f * 1.e-3
    mus = compute_mus(df_ci, x_mc, x_nrlee, v_species, f, df_low, df_high)
    
    title_str = make_title_str(df, x_species, y_species, lnames, mus, nc_only)
    
    x_val = "mu_" + x_species + "_value"
    y_val = "mu_" + y_species + "_value"

    x_err = "mu_" + x_species + "_variability of sample"
    y_err = "mu_" + y_species + "_variability of sample"

    if nc_only:
        sns.lmplot(data=df_nc, x=x_val, y=y_val, line_kws={'color': 'black', 'linestyle': "dotted", "label": "Data fit"})
    else:
        sns.lmplot(data=df, x=x_val, y=y_val, line_kws={'color': 'black', 'linestyle': "dotted", "label": "Data fit"})

    plt.errorbar(df_nc[x_val], df_nc[y_val], xerr=df_nc[x_err], yerr=df_nc[y_err], ls='none', color = 'red', marker='o', label='NC')
    if not nc_only:
        plt.errorbar(df_cc[x_val], df_cc[y_val], xerr=df_cc[x_err], yerr=df_cc[y_err], ls='none', color = 'blue', marker='o', label='CC')

    plt.errorbar(df_ci[x_val], df_ci[y_val], xerr=df_ci[x_err], yerr=df_ci[y_err], ls='none', color = '#ff00ff', marker='o', label='CI')
    plt.errorbar(df_earth[x_val], df_earth[y_val], xerr=df_earth[x_err], yerr=df_earth[y_err], ls='none', color = '#00ff00', marker='o', label='Bulk Earth')

    plt.plot(mus[0], mus[1], label="+/- NRLEE", color='black')

    plt.xlabel('$\\mu${:s}'.format(lnames[v_species[0]]))
    plt.ylabel('$\\mu${:s}'.format(lnames[v_species[2]]))

    if show_title:
        plt.title(title_str, fontsize=10)
    sns.despine(top=False, right=False)

    plt.legend(loc='upper left')

    if figure_name:
        plt.savefig(figure_name, bbox_inches='tight')
        

x_species = widgets.Text(value='ca48',placeholder='Type something',
                         description='x_species:', disabled=False)
y_species = widgets.Text(value='ti50',placeholder='Type something',
                         description='y_species:', disabled=False)
f_4 = widgets.BoundedFloatText(value=1., min=0, max = 1, step=0.01,
                              description='f_4:', disabled=False)
df_f_low = widgets.BoundedFloatText(value=-2, min=-10, max = 0, step=0.01, 
                                 description='df_f_low:', disabled=False)
df_f_high = widgets.BoundedFloatText(value=0, min=0, max = 10, step=0.01, 
                                  description='df_f_high:', disabled=False)
show_title = widgets.Checkbox(value=True, description='Show title', disabled=False, 
                              indent=True)
nc_only = widgets.Checkbox(value=False, description='NC fit only', disabled=False, 
                           indent=True)
figure_name = widgets.Text(value=None,placeholder='Type something',
                           description='figure name:', disabled=False)
 
out = interactive_output(plot_nc_from_ci, {'x_species': x_species, 
                                           'y_species': y_species, 'f_4': f_4,
                                           'df_f_low': df_f_low, 'df_f_high': df_f_high,
                                           'show_title': show_title, 'nc_only': nc_only,
                                           'figure_name': figure_name})

display(HBox([x_species, y_species]), HBox([f_4, figure_name]),
        HBox([df_f_low, df_f_high]),  
        HBox([show_title, nc_only]), out)

Output()

# CAI-like Material

In [20]:
def plot_cai(x_species, y_species, f_4, f_cai_4, g_x_n, g_x_nbar, g_y_n, g_y_nbar, cc_only, figure_name):

    v_species = [x_species, norm_dict[x_species], y_species, norm_dict[y_species]]
    
    f = f_4 * 1.e-4
    f_cai = f_cai_4 * 1.e-4
    mu_x, mu_y = compute_mu_cai(df_ci, x_mc, x_nrlee, v_species, f, f_cai,
                         g_x_n, g_x_nbar, g_y_n, g_y_nbar)
    
    v_mu_x = np.array([0, mu_x])
    v_mu_y = np.array([0, mu_y])
        
    x_val = "mu_" + x_species + "_value"
    y_val = "mu_" + y_species + "_value"

    x_err = "mu_" + x_species + "_variability of sample"
    y_err = "mu_" + y_species + "_variability of sample"
    
    v_mu_x *=(1 + df_ci[x_val] / 1.e6) 
    v_mu_x += df_ci[x_val]
    
    v_mu_y *=(1 + df_ci[y_val] / 1.e6) 
    v_mu_y += df_ci[y_val]

    sns.lmplot(data=df_cc, x=x_val, y=y_val, line_kws={'color': 'black',
                                                    'linestyle': 'dotted',
                                                    "label": "Data fit"})

    plt.errorbar(df_cc[x_val], df_cc[y_val], xerr=df_cc[x_err], yerr=df_cc[y_err],
                 ls='none', color = 'blue', marker='o', label='CC')

    plt.errorbar(df_ci[x_val], df_ci[y_val], xerr=df_ci[x_err], yerr=df_ci[y_err],
                 ls='none', color = '#ff00ff', marker='o', label='CI')
    
    if not cc_only:
        plt.errorbar(df_nc[x_val], df_nc[y_val], xerr=df_nc[x_err], yerr=df_nc[y_err], ls='none', color = 'red', marker='o', label='NC')
        plt.errorbar(df_earth[x_val], df_earth[y_val], xerr=df_earth[x_err],
                     yerr=df_earth[y_err], ls='none', color = '#00ff00', marker='o',
                     label='Bulk Earth')

    plt.plot(v_mu_x, v_mu_y, ':o', label="CAI", color='black')

    plt.xlabel('$\\mu${:s}'.format(lnames[v_species[0]]))
    plt.ylabel('$\\mu${:s}'.format(lnames[v_species[2]]))

    sns.despine(top=False, right=False)

    plt.legend()
        
    if figure_name:
        plt.savefig(figure_name, bbox_inches='tight')

x_species = widgets.Text(value='ca48',placeholder='Type something',
                         description='x_species:', disabled=False)
y_species = widgets.Text(value='ti50',placeholder='Type something',
                         description='y_species:', disabled=False)
f_4 = widgets.BoundedFloatText(value=1., min=0, max = 1, step=0.00001,
                              description='f_4:', disabled=False)
f_cai_4 = widgets.BoundedFloatText(value=1.01, min=0, max = 2, step=0.0001, 
                                 description='f_cai_4:', disabled=False)
g_x_n = widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_x_n:', disabled=False)
g_x_nbar= widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_x_nbar:', disabled=False)
g_y_n = widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_y_n:', disabled=False)
g_y_nbar= widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_y_nbar:', disabled=False)
cc_only = widgets.Checkbox(value=False, description='CC fit only', disabled=False, 
                           indent=True)
figure_name = widgets.Text(value=None,placeholder='Type something',
                           description='figure name:', disabled=False)
 


out2 = interactive_output(plot_cai, {'x_species': x_species,
                                               'y_species': y_species,
                                               'f_4': f_4, 'f_cai_4': f_cai_4,
                                               'g_x_n': g_x_n,
                                               'g_x_nbar': g_x_nbar, 'g_y_n': g_y_n,
                                               'g_y_nbar': g_y_nbar, 'cc_only': cc_only,
                                               'figure_name': figure_name})

display(HBox([x_species, y_species]), HBox([f_4, f_cai_4]),  
        HBox([g_x_n, g_x_nbar]),
        HBox([g_y_n, g_y_nbar]), 
        HBox([cc_only, figure_name]), out2)

Output()

# Admixture of CAI-like Material

In [21]:
def plot_cai_admixture(x_species, y_species, f_4, f_cai_4, fs_max,
                       g_x_n, g_x_nbar, g_y_n, g_y_nbar, cc_only, figure_name):

    v_species = [x_species, norm_dict[x_species], y_species, norm_dict[y_species]]
    
    f = f_4 * 1.e-4
    f_cai = f_cai_4 * 1.e-4
    mu_x, mu_y = compute_mu_cai(df_ci, x_mc, x_nrlee, v_species, f, f_cai,
                         g_x_n, g_x_nbar, g_y_n, g_y_nbar)
    
    print(mu_x, mu_y)
    
    fs = np.linspace(0, fs_max, 10)
    v_mu_x = mu_x * (fs / (1 + fs))
    v_mu_y = mu_y * (fs / (1 + fs))
        
    x_val = "mu_" + x_species + "_value"
    y_val = "mu_" + y_species + "_value"

    x_err = "mu_" + x_species + "_variability of sample"
    y_err = "mu_" + y_species + "_variability of sample"
    
    v_mu_x *=(1 + df_ci[x_val] / 1.e6) 
    v_mu_x += df_ci[x_val]
    
    v_mu_y *=(1 + df_ci[y_val] / 1.e6) 
    v_mu_y += df_ci[y_val]

    sns.lmplot(data=df_cc, x=x_val, y=y_val, line_kws={'color': 'black',
                                                    'linestyle': "dotted",
                                                    "label": "Data fit"})

    plt.errorbar(df_cc[x_val], df_cc[y_val], xerr=df_cc[x_err], yerr=df_cc[y_err],
                 ls='none', color = 'blue', marker='o', label='CC')

    plt.errorbar(df_ci[x_val], df_ci[y_val], xerr=df_ci[x_err], yerr=df_ci[y_err],
                 ls='none', color = '#ff00ff', marker='o', label='CI')
    
    if not cc_only:
        plt.errorbar(df_nc[x_val], df_nc[y_val], xerr=df_nc[x_err], yerr=df_nc[y_err], ls='none', color = 'red', marker='o', label='NC')
        plt.errorbar(df_earth[x_val], df_earth[y_val], xerr=df_earth[x_err],
                     yerr=df_earth[y_err], ls='none', color = '#00ff00', marker='o',
                     label='Bulk Earth')

    plt.plot(v_mu_x, v_mu_y, label="CAI fit", color='black')

    plt.xlabel('$\\mu${:s}'.format(lnames[v_species[0]]))
    plt.ylabel('$\\mu${:s}'.format(lnames[v_species[2]]))

    sns.despine(top=False, right=False)

    plt.legend()
        
    if figure_name:
        plt.savefig(figure_name, bbox_inches='tight')

x_species = widgets.Text(value='ca48',placeholder='Type something',
                         description='x_species:', disabled=False)
y_species = widgets.Text(value='ti50',placeholder='Type something',
                         description='y_species:', disabled=False)
f_4 = widgets.BoundedFloatText(value=1., min=0, max = 1, step=0.00001,
                              description='f_4:', disabled=False)
f_cai_4 = widgets.BoundedFloatText(value=1.01, min=0, max = 2, step=0.01, 
                                 description='f_cai_4:', disabled=False)
fs_max = widgets.BoundedFloatText(value=0.2, min=0, max = 10, step=0.1, 
                                 description='fs_max:', disabled=False)
g_x_n = widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_x_n:', disabled=False)
g_x_nbar= widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_x_nbar:', disabled=False)
g_y_n = widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_y_n:', disabled=False)
g_y_nbar= widgets.BoundedFloatText(value=1, min=0, max = 1, step=0.001, 
                                 description='g_y_nbar:', disabled=False)
cc_only = widgets.Checkbox(value=False, description='CC fit only', disabled=False, 
                           indent=True)
figure_name = widgets.Text(value=None,placeholder='Type something',
                           description='figure name:', disabled=False)
 


out2 = interactive_output(plot_cai_admixture, {'x_species': x_species,
                                               'y_species': y_species,
                                               'f_4': f_4, 'f_cai_4': f_cai_4,
                                               'fs_max': fs_max, 'g_x_n': g_x_n,
                                               'g_x_nbar': g_x_nbar, 'g_y_n': g_y_n,
                                               'g_y_nbar': g_y_nbar, 'cc_only': cc_only,
                                               'figure_name': figure_name})

display(HBox([x_species, y_species]), HBox([f_4, f_cai_4]),  
        HBox([fs_max]),
        HBox([g_x_n, g_x_nbar]),
        HBox([g_y_n, g_y_nbar]), 
        HBox([cc_only, figure_name]), out2)

Output()